In [1]:
from datasets import load_dataset
import pandas as pd
import json
import random
from typing import Dict, List

In [2]:
# Turkish MMLU veri setini yükle
dataset = load_dataset("alibayram/turkish_mmlu")
print(f"Veri seti yapısı: {dataset}")
print(f"Toplam örnek sayısı: {len(dataset['test'])}")

Veri seti yapısı: DatasetDict({
    train: Dataset({
        features: ['bolum', 'konu', 'soru', 'cevap', 'aciklama', 'secenekler'],
        num_rows: 232136
    })
    test: Dataset({
        features: ['bolum', 'konu', 'soru', 'cevap', 'aciklama', 'secenekler'],
        num_rows: 40966
    })
    mmlu: Dataset({
        features: ['bolum', 'konu', 'soru', 'cevap', 'aciklama', 'secenekler'],
        num_rows: 6200
    })
})
Toplam örnek sayısı: 40966


In [3]:
# İlk birkaç örneği incele
sample = dataset['test'][0]
print("Örnek veri yapısı:")
for key, value in sample.items():
    print(f"{key}: {value}")
    
# Tüm subject'leri gör
if 'subject' in sample:
    all_subjects = set(item['subject'] for item in dataset['test'] if 'subject' in item)
    print(f"\nMevcut subject'ler: {sorted(all_subjects)}")
else:
    # Subject bilgisi yoksa, tüm örnekleri yazdır
    print("\nSubject bilgisi bulunamadı, veri yapısını kontrol ediyoruz...")
    print(f"Keys: {sample.keys()}")

# Filtrelemeden ÖNCE tüm bölümleri ve sayılarını göster
print("\n" + "="*50)
print("FİLTRELEMEDEN ÖNCE - TÜM BÖLÜMLER VE SAYILARI:")
print("="*50)

# Tüm bölümleri ve sayılarını say
bolum_counts = {}
for item in dataset['test']:
    bolum = item.get('bolum') or 'None/Boş'
    bolum_counts[bolum] = bolum_counts.get(bolum, 0) + 1

# Sayıya göre sırala ve göster
for bolum, count in sorted(bolum_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {bolum}: {count} örnek")

print(f"\nToplam örnek sayısı: {len(dataset['test'])}")
print(f"Toplam farklı bölüm sayısı: {len(bolum_counts)}")

Örnek veri yapısı:
bolum: Çocuk Gelişimi
konu: Hasta Çocukların Gelişimi ve Eğitimi
soru: Normal insan görüşü, çevresel alanı da eklemek koşuluyla yatay olarak kaç derecedir?
cevap: 3
aciklama: None
secenekler: ['45', '90', '120', '180', '360']

Subject bilgisi bulunamadı, veri yapısını kontrol ediyoruz...
Keys: dict_keys(['bolum', 'konu', 'soru', 'cevap', 'aciklama', 'secenekler'])

FİLTRELEMEDEN ÖNCE - TÜM BÖLÜMLER VE SAYILARI:
  Kim 500 Milyar İster: 12968 örnek
  YGS Denemeleri: 4154 örnek
  KPSS Denemeleri: 2674 örnek
  Çocuk Gelişimi: 2597 örnek
  TUS: 1510 örnek
  Laborant ve Veteriner Sağlık: 1485 örnek
  Dış Ticaret: 1173 örnek
  Siyer: 1067 örnek
  Sosyal Hizmetler: 935 örnek
  Medya ve İletişim: 898 örnek
  Kültürel Miras ve Turizm: 859 örnek
  Özel Koruma ve Güvenlik: 759 örnek
  Tarım: 586 örnek
  Sağlık Kurumları İşletmeciliği: 575 örnek
  Tarih: 571 örnek
  Radyo ve Televizyon Programcılığı: 531 örnek
  Felsefe: 502 örnek
  İşletme Yönetimi: 501 örnek
  Lojistik: 474 örn

In [4]:
# Mantık, çıkarım ve problem çözme gerektiren konular
# Bölüm bazında direkt dahil edilecekler (tam eşleşme)
target_bolumler_exact = [
    'YGS Denemeleri',
    'KPSS',
    'KPSS Denemeleri',
    'Felsefe',
    'İktisat',
    'İşletme Yönetimi',
    'Sosyoloji',
    'Dış Ticaret',
    'TUS',
    'Yönetim Bİlişim Sistemleri',
    'Uluslararası Ticaret ve Lojistik Yöneticiliği',
    'İşletme'
]

# Konu/bölüm içinde aranacak kelimeler
target_keywords = [
    'matematik', 'mathematics', 'math',
    'fizik', 'physics',
    'mantık', 'logic', 'logik',
    'bilgisayar', 'computer', 'cs',
    'mühendislik', 'engineering',
    'istatistik', 'statistics',
    'kimya', 'chemistry',
    'geometri', 'geometry',
    'analiz', 'analysis',
    'problem', 'çözme',
    'hesaplama', 'calculation',
    'denklem', 'equation',
    'cebir', 'algebra'
]

def filter_logical_subjects(example):
    """Mantıksal çıkarım gerektiren konuları filtrele"""
    bolum = example.get('bolum') or ''
    konu = example.get('konu') or ''
    soru = example.get('soru', '')
    secenekler = example.get('secenekler', [])
    
    # Uzunluk kontrolü: Soru + tüm secenekler toplamı en az 200 karakter olmalı
    soru_uzunlugu = len(soru)
    secenekler_toplam_uzunlugu = sum(len(sec) for sec in secenekler)
    toplam_uzunluk = soru_uzunlugu + secenekler_toplam_uzunlugu
    
    if toplam_uzunluk < 200:
        return False
    
    # None kontrolü: eğer string değilse boş string'e çevir
    bolum_str = bolum.lower() if isinstance(bolum, str) else ''
    konu_str = konu.lower() if isinstance(konu, str) else ''
    
    # 1. Bölüm tam eşleşme kontrolü
    if bolum in target_bolumler_exact:
        return True
    
    # 2. Bölüm veya konu içinde hedef kelimelerden biri var mı kontrol et
    return any(keyword in bolum_str or keyword in konu_str for keyword in target_keywords)

# Filtreleme işlemi
filtered_dataset = dataset['test'].filter(filter_logical_subjects)
print(f"Filtreleme sonrası örnek sayısı: {len(filtered_dataset)}")

# Filtrelenmiş bölümleri göster
filtered_bolumler = set(item.get('bolum', 'unknown') for item in filtered_dataset)
print(f"\nFiltrelenmiş bölümler: {sorted(filtered_bolumler)}")

Filtreleme sonrası örnek sayısı: 7925

Filtrelenmiş bölümler: ['Dış Ticaret', 'Felsefe', 'KPSS', 'KPSS Denemeleri', 'Laborant ve Veteriner Sağlık', 'Marka İletişimi', 'Menkul Kıymetler ve Sermaye Piyasası', 'Muhasebe ve Vergi Uygulamaları', 'Sosyoloji', 'TUS', 'Tarım', 'Uluslar Arası İlişkiler', 'YGS Denemeleri', 'Yönetim Bİlişim Sistemleri', 'İktisat', 'İşletme', 'İşletme Yönetimi']


In [5]:
# Filtrelenmiş veri setinin detaylı metriklerini incele
import statistics

print("="*60)
print("FİLTRELENMİŞ VERİ SETİ METRİKLERİ")
print("="*60)

# Metrikleri hesapla
soru_uzunluklari = []
secenek_uzunluklari = []
toplam_secenek_uzunluklari = []
soru_basliklari = []
secenek_sayilari = []
cevap_dagilimi = {'A': 0, 'B': 0, 'C': 0, 'D': 0, 'E': 0}

for item in filtered_dataset:
    soru = item.get('soru', '')
    secenekler = item.get('secenekler', [])
    cevap_index = item.get('cevap', 0)
    
    # Soru uzunluğu
    soru_uzunluklari.append(len(soru))
    
    # Secenek uzunlukları
    secenek_uzunluklari.extend([len(sec) for sec in secenekler])
    toplam_secenek_uzunluklari.append(sum(len(sec) for sec in secenekler))
    
    # Secenek sayısı
    secenek_sayilari.append(len(secenekler))
    
    # Cevap dağılımı
    if isinstance(cevap_index, int) and 0 <= cevap_index < len(secenekler):
        cevap_harf = chr(65 + cevap_index)
        if cevap_harf in cevap_dagilimi:
            cevap_dagilimi[cevap_harf] += 1

# İstatistikleri yazdır
print(f"\n📊 TEMEL İSTATİSTİKLER:")
print(f"  Toplam örnek sayısı: {len(filtered_dataset)}")

print(f"\n📝 SORU METRİKLERİ:")
print(f"  Ortalama soru uzunluğu: {statistics.mean(soru_uzunluklari):.1f} karakter")
print(f"  Medyan soru uzunluğu: {statistics.median(soru_uzunluklari):.1f} karakter")
print(f"  Min soru uzunluğu: {min(soru_uzunluklari)} karakter")
print(f"  Max soru uzunluğu: {max(soru_uzunluklari)} karakter")

print(f"\n🔤 SECENEK METRİKLERİ:")
print(f"  Ortalama secenek uzunluğu: {statistics.mean(secenek_uzunluklari):.1f} karakter")
print(f"  Medyan secenek uzunluğu: {statistics.median(secenek_uzunluklari):.1f} karakter")
print(f"  Ortalama toplam secenek uzunluğu (soru başına): {statistics.mean(toplam_secenek_uzunluklari):.1f} karakter")

print(f"\n🔢 SECENEK SAYISI DAĞILIMI:")
secenek_sayi_dagilimi = {}
for sayi in secenek_sayilari:
    secenek_sayi_dagilimi[sayi] = secenek_sayi_dagilimi.get(sayi, 0) + 1
for sayi, count in sorted(secenek_sayi_dagilimi.items()):
    print(f"  {sayi} şıklı sorular: {count} adet ({count/len(filtered_dataset)*100:.1f}%)")

print(f"\n✅ CEVAP DAĞILIMI:")
for harf, count in sorted(cevap_dagilimi.items()):
    yuzde = count / len(filtered_dataset) * 100
    print(f"  {harf}: {count} adet ({yuzde:.1f}%)")

print(f"\n📂 BÖLÜM DAĞILIMI:")
bolum_dagilimi = {}
for item in filtered_dataset:
    bolum = item.get('bolum', 'unknown')
    bolum_dagilimi[bolum] = bolum_dagilimi.get(bolum, 0) + 1

for bolum, count in sorted(bolum_dagilimi.items(), key=lambda x: x[1], reverse=True):
    yuzde = count / len(filtered_dataset) * 100
    print(f"  {bolum}: {count} adet ({yuzde:.1f}%)")

print(f"\n📈 TOPLAM METNİN UZUNLUĞU (Soru + Tüm Secenekler):")
toplam_metin_uzunluklari = [soru_uzunluklari[i] + toplam_secenek_uzunluklari[i] 
                            for i in range(len(soru_uzunluklari))]
print(f"  Ortalama: {statistics.mean(toplam_metin_uzunluklari):.1f} karakter")
print(f"  Medyan: {statistics.median(toplam_metin_uzunluklari):.1f} karakter")
print(f"  Min: {min(toplam_metin_uzunluklari)} karakter")
print(f"  Max: {max(toplam_metin_uzunluklari)} karakter")


FİLTRELENMİŞ VERİ SETİ METRİKLERİ

📊 TEMEL İSTATİSTİKLER:
  Toplam örnek sayısı: 7925

📝 SORU METRİKLERİ:
  Ortalama soru uzunluğu: 282.3 karakter
  Medyan soru uzunluğu: 237.0 karakter
  Min soru uzunluğu: 12 karakter
  Max soru uzunluğu: 1371 karakter

🔤 SECENEK METRİKLERİ:
  Ortalama secenek uzunluğu: 20.4 karakter
  Medyan secenek uzunluğu: 10.0 karakter
  Ortalama toplam secenek uzunluğu (soru başına): 101.9 karakter

🔢 SECENEK SAYISI DAĞILIMI:
  5 şıklı sorular: 7925 adet (100.0%)

✅ CEVAP DAĞILIMI:
  A: 1319 adet (16.6%)
  B: 1490 adet (18.8%)
  C: 1707 adet (21.5%)
  D: 1712 adet (21.6%)
  E: 1697 adet (21.4%)

📂 BÖLÜM DAĞILIMI:
  YGS Denemeleri: 3574 adet (45.1%)
  KPSS Denemeleri: 1709 adet (21.6%)
  TUS: 721 adet (9.1%)
  Dış Ticaret: 536 adet (6.8%)
  Felsefe: 250 adet (3.2%)
  İşletme Yönetimi: 247 adet (3.1%)
  KPSS: 204 adet (2.6%)
  Sosyoloji: 198 adet (2.5%)
  İktisat: 192 adet (2.4%)
  Yönetim Bİlişim Sistemleri: 166 adet (2.1%)
  Menkul Kıymetler ve Sermaye Piyasası:

In [ ]:
def format_for_grpo(example: Dict) -> Dict:
    """
    Turkish MMLU formatını GRPO formatına dönüştür (Basitleştirilmiş Versiyon).
    Modelden beklenen format: 
    Açıklama: ...
    Cevap: ...
    """
    soru = example.get('soru', '')
    secenekler = example.get('secenekler', [])
    cevap_index = example.get('cevap', 0)  # 0-indexed index
    
    # Cevap index'ini harfe çevir (A, B, C, D, E)
    if isinstance(cevap_index, int) and 0 <= cevap_index < len(secenekler):
        correct_answer_letter = chr(65 + cevap_index)
    else:
        correct_answer_letter = 'A'
    
    # Şıkları formatla
    choices_text = "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(secenekler)])
    
    # System Prompt (Artık User Prompt'un içinde olacak çünkü model system desteklemiyor)
    # Ayrıca XML formatı yerine basit format kullanıyoruz.
    instruction = """Aşağıdaki soruyu dikkatlice adım adım düşünerek çöz. Düşünce sürecini açıklama kısmına, cevabını cevap kısmına yaz.
Çözüm formatı:
Açıklama: Burada düşünme sürecini, mantıksal çıkarımlarını ve akıl yürütmeni yaz
Cevap: Sadece doğru şıkkın harfini yaz (A, B, C, D veya E)

Soru:
"""
    
    full_user_content = f"{instruction}{soru}\n\n{choices_text}"
    
    return {
        'messages': [
            {'role': 'user', 'content': full_user_content}
        ],
        'correct_answer': correct_answer_letter,
        'bolum': example.get('bolum', 'unknown'),
        'konu': example.get('konu', 'unknown'),
        'original_answer_index': cevap_index
    }

# Test: Bir örnek dönüştür
sample_formatted = format_for_grpo(filtered_dataset[0])
print("Formatlanmış örnek:")
print(json.dumps(sample_formatted, ensure_ascii=False, indent=2))


Formatlanmış örnek:
{
  "messages": [
    {
      "role": "user",
      "content": "Aşağıdaki soruyu dikkatlice adım adım düşünerek çöz. Düşünce sürecini açıklama kısmına, cevabını cevap kısmına yaz.\nÇözümünü şu formatta sun:\n\nAçıklama: [Burada düşünme sürecini, mantıksal çıkarımlarını ve akıl yürütmeni yaz]\nCevap: [Sadece doğru şıkkın harfini yaz (A, B, C, D veya E)]\n\nSoru:\nI. Çocukların bakım ve beslenmesine özen gösterilmiştir. II. Çocuklar 5-7 yaşına kadar bebek olarak görülmüştür. III. Doğum ve bebek ölüm oranı düşüktür. IV. Çocuklar, 6 yaşına kadar ailenin bir üyesi olarak algılanmamıştır. Ortaçağdaki çocuk anlayışı ile ilgili yukarıdaki ifadelerden hangileri yanlıştır?\n\nA. Yalnız II\nB. I ve II\nC. I ve III\nD. II ve IV\nE. III ve IV"
    }
  ],
  "correct_answer": "C",
  "bolum": "Dış Ticaret",
  "konu": "Genel Matematik",
  "original_answer_index": 2
}


In [7]:
# Tüm filtrelenmiş veriyi GRPO formatına dönüştür
# Sadece gereksiz kolonları kaldır, bolum ve konu'yu tut
columns_to_remove = [col for col in filtered_dataset.column_names if col not in ['bolum', 'konu']]
formatted_dataset = filtered_dataset.map(
    format_for_grpo, 
    remove_columns=columns_to_remove
)
print(f"Formatlanmış veri sayısı: {len(formatted_dataset)}")


Map:   0%|          | 0/7925 [00:00<?, ? examples/s]

Formatlanmış veri sayısı: 7925


In [8]:
# Veriyi rastgele karıştır
seed = 42
random.seed(seed)

# Veriyi listeye çevir ve karıştır
data_list = list(formatted_dataset)
random.shuffle(data_list)

# Train (1000 örnek) ve Test (500 örnek) olarak ayır
train_size = 1000
test_size = 500

train_data = data_list[:train_size]
test_data = data_list[train_size:train_size + test_size]

print(f"Train seti: {len(train_data)} örnek")
print(f"Test seti: {len(test_data)} örnek")
print(f"Toplam kullanılan: {len(train_data) + len(test_data)} örnek")


Train seti: 1000 örnek
Test seti: 500 örnek
Toplam kullanılan: 1500 örnek


In [9]:
from datasets import Dataset

# HuggingFace Dataset formatına çevir
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Train dataset: {train_dataset}")
print(f"Test dataset: {test_dataset}")

# İsteğe bağlı: Disk'e kaydet (sonradan kullanmak için)
train_dataset.save_to_disk("data/train_dataset")
test_dataset.save_to_disk("data/test_dataset")


Train dataset: Dataset({
    features: ['bolum', 'konu', 'messages', 'correct_answer', 'original_answer_index'],
    num_rows: 1000
})
Test dataset: Dataset({
    features: ['bolum', 'konu', 'messages', 'correct_answer', 'original_answer_index'],
    num_rows: 500
})


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
# Train seti istatistikleri
print("=== TRAIN SETİ İSTATİSTİKLERİ ===")
train_bolumler = {}
for item in train_data:
    bolum = item.get('bolum', 'unknown')
    train_bolumler[bolum] = train_bolumler.get(bolum, 0) + 1

print(f"Train setindeki bölüm dağılımı:")
for bolum, count in sorted(train_bolumler.items(), key=lambda x: x[1], reverse=True):
    print(f"  {bolum}: {count}")

print(f"\n=== TEST SETİ İSTATİSTİKLERİ ===")
test_bolumler = {}
for item in test_data:
    bolum = item.get('bolum', 'unknown')
    test_bolumler[bolum] = test_bolumler.get(bolum, 0) + 1

print(f"Test setindeki bölüm dağılımı:")
for bolum, count in sorted(test_bolumler.items(), key=lambda x: x[1], reverse=True):
    print(f"  {bolum}: {count}")

# Örnek bir train örneği göster
print(f"\n=== TRAIN SETİ ÖRNEĞİ ===")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


=== TRAIN SETİ İSTATİSTİKLERİ ===
Train setindeki bölüm dağılımı:
  YGS Denemeleri: 432
  KPSS Denemeleri: 203
  TUS: 99
  Dış Ticaret: 77
  Felsefe: 34
  İktisat: 29
  Yönetim Bİlişim Sistemleri: 27
  Sosyoloji: 27
  KPSS: 25
  İşletme Yönetimi: 24
  Menkul Kıymetler ve Sermaye Piyasası: 6
  Laborant ve Veteriner Sağlık: 6
  Muhasebe ve Vergi Uygulamaları: 5
  İşletme: 4
  Tarım: 1
  Uluslar Arası İlişkiler: 1

=== TEST SETİ İSTATİSTİKLERİ ===
Test setindeki bölüm dağılımı:
  YGS Denemeleri: 218
  KPSS Denemeleri: 111
  TUS: 45
  Dış Ticaret: 38
  Felsefe: 17
  Yönetim Bİlişim Sistemleri: 15
  KPSS: 14
  İktisat: 12
  İşletme Yönetimi: 12
  Sosyoloji: 11
  Menkul Kıymetler ve Sermaye Piyasası: 3
  Uluslar Arası İlişkiler: 2
  Laborant ve Veteriner Sağlık: 1
  Muhasebe ve Vergi Uygulamaları: 1

=== TRAIN SETİ ÖRNEĞİ ===
{
  "bolum": "TUS",
  "konu": null,
  "messages": [
    {
      "content": "Aşağıdaki soruyu dikkatlice adım adım düşünerek çöz. Düşünce sürecini açıklama kısmına, ceva

In [ ]:
def validate_format(example):
    """Formatın doğru olduğunu kontrol et"""
    messages = example.get('messages', [])
    correct_answer = example.get('correct_answer', '')
    
    assert len(messages) == 1, "Sadece User mesajı olmalı (System prompt user içine gömüldü)"
    assert messages[0]['role'] == 'user', "Mesaj user olmalı"
    assert "Açıklama:" in messages[0]['content'], "User mesajında format talimatı olmalı"
    assert correct_answer in ['A', 'B', 'C', 'D', 'E'], f"Doğru cevap A-E arası olmalı, bulundu: {correct_answer}"
    
    return True

# Tüm veriyi kontrol et
print("Format doğrulaması yapılıyor...")
for i, item in enumerate(train_data[:10]):  # İlk 10'unu kontrol et
    try:
        validate_format(item)
        print(f"✓ Örnek {i+1}: Geçerli")
    except AssertionError as e:
        print(f"✗ Örnek {i+1}: HATA - {e}")

print("\n✓ Veri hazırlığı tamamlandı!")
print(f"✓ Train seti: {len(train_dataset)} örnek")
print(f"✓ Test seti: {len(test_dataset)} örnek")


Format doğrulaması yapılıyor...
✓ Örnek 1: Geçerli
✓ Örnek 2: Geçerli
✓ Örnek 3: Geçerli
✓ Örnek 4: Geçerli
✓ Örnek 5: Geçerli
✓ Örnek 6: Geçerli
✓ Örnek 7: Geçerli
✓ Örnek 8: Geçerli
✓ Örnek 9: Geçerli
✓ Örnek 10: Geçerli

✓ Veri hazırlığı tamamlandı!
✓ Train seti: 1000 örnek
✓ Test seti: 500 örnek


: 